# Inspection of the reassembled full glycoporteins dataset (from Mifsud et al.)

In [1]:
import pandas as pd
import os

os.getcwd()

'/Users/finnweikert/Desktop/Dessimoz-Lab/ESM3di/Viral-FoldTree'

In [2]:
# load fasta file into pandas dataframe
fasta_file = "data/flaviviridae_fullglyco/refolded_fullglyco_aa.fas"

# load the fasta format correctly into a pandas dataframe lines starting with '>' as the header and the next line as the sequence
def load_fasta_to_dataframe(fasta_file):
    with open(fasta_file, "r") as f:
        lines = f.readlines()

    data = []
    for i in range(0, len(lines), 2):
        header = lines[i].strip()
        sequence = lines[i + 1].strip()
        data.append([header, sequence])

    df = pd.DataFrame(data, columns=["Header", "Sequence"])
    return df

df = load_fasta_to_dataframe(fasta_file)

In [3]:
df['seq_length'] = df['Sequence'].apply(len)

print(df['seq_length'].describe())

count    626.000000
mean     313.910543
std      115.488936
min       98.000000
25%      194.250000
50%      309.000000
75%      396.000000
max      505.000000
Name: seq_length, dtype: float64


In [4]:
df['Header']

0      >FJAF_Cnidaria_flavivirus_E_278-749_AlphaFold.pdb
1      >FJAF_Crangon_crangon_flavivirus_orf_1_E_310-7...
2      >FJAF_Gammarus_chevreuxi_flavivirus_isolate_10...
3      >FJAF_Gammarus_pulex_flavivirus_isolate_GPCM_o...
4      >FJAF_Harrimaniidae_flavivirus_E_257-629_Alpha...
                             ...                        
621          >PLPV_Zikole_virus_E2_710-948_AlphaFold.pdb
622    >PLUN_Parasteatoda_tepidariorum_associated_fla...
623    >PLUN_Shayang_spider_virus_4_strain_SYZZ-1_E_1...
624    >PLUN_Xiamen_fanray_pesti-like_virus_strain_XM...
625    >PLUN_Xinzhou_spider_virus_3_strain_XZZZ-3_E_1...
Name: Header, Length: 626, dtype: object

# Robinson-Foulds (RF) distance between trees

In [4]:
import re
import dendropy
from dendropy.calculate import treecompare

In [15]:
import dendropy
from dendropy.calculate import treecompare

def load_and_clean_newick(file_path):
    """Reads a tree file and removes '.pdb' from taxon identifiers."""
    with open(file_path, "r") as f:
        content = f.read()
    return content.replace(".pdb", "")

glycoproteins = ["E", "E1", "E2"]

for prot in glycoproteins:
    # 1. Establish a shared namespace
    tns = dendropy.TaxonNamespace()

    # 2. Read clean Newick content
    misfud_foldtree = load_and_clean_newick(f"Foldtree/{prot.lower()}_foldtree_struct_tree.PP.nwk")
    misfud_aa_only = load_and_clean_newick(f"fn_3di_trees/refolded_fullglyco_{prot}_AA_famsa.fas.treefile")
    esm3di_unrooted = load_and_clean_newick(f"results/flaviviridae_fullglyco_{prot}/foldtree_struct_tree.PP.nwk")

    misfud_3di_our_pipeline = load_and_clean_newick(f"results/misfud_fastas/flaviviridae_fullglyco_{prot}/foldtree_struct_tree.PP.nwk")
    misfud_foldtree_reproduced = load_and_clean_newick(f"Foldtree_reproduced/{prot.lower()}_foldtree_struct_tree.PP.nwk")


    # 3. Parse trees into DendroPy and force them to be treated as unrooted
    tree1 = dendropy.Tree.get(
        data=misfud_foldtree, 
        schema="newick", 
        taxon_namespace=tns,
        rooting="force-unrooted"
    )

    tree2 = dendropy.Tree.get(
        data=misfud_aa_only, 
        schema="newick", 
        taxon_namespace=tns,
        rooting="force-unrooted"
    )

    tree3 = dendropy.Tree.get(
        data=esm3di_unrooted, 
        schema="newick",
        taxon_namespace=tns,
        rooting="force-unrooted"
    )

    tree4 = dendropy.Tree.get(
        data=misfud_3di_our_pipeline, 
        schema="newick",
        taxon_namespace=tns,
        rooting="force-unrooted"
    )

    tree5 = dendropy.Tree.get(
        data=misfud_foldtree_reproduced,
        schema="newick",
        taxon_namespace=tns,
        rooting="force-unrooted"
    )

    # 4. Collapse degree-2 basal node if present & mark unrooted
    trees = [tree1, tree2, tree3, tree4, tree5]
    leaf_counts = [len(t.leaf_nodes()) for t in trees]
    assert len(set(leaf_counts)) == 1, f"Taxa count mismatch across trees! {leaf_counts}"

    for tree in trees:
        tree.collapse_basal_bifurcation()
        tree.is_rooted = False
        

    # 5. Encode bipartitions using the shared namespace
    tree1.encode_bipartitions()
    tree2.encode_bipartitions()
    tree3.encode_bipartitions()
    tree4.encode_bipartitions()
    tree5.encode_bipartitions()

    # 6. Calculate unweighted Robinson-Foulds distance
    rf_dist_foldtree = treecompare.symmetric_difference(tree5, tree3)
    rf_dist_aa_only = treecompare.symmetric_difference(tree2, tree3)
    rf_dist_foldree_3di_our_pipeline = treecompare.symmetric_difference(tree1, tree4)
    rf_dist_esm3di_3di_our_pipeline = treecompare.symmetric_difference(tree3, tree4)
    rf_dist_foldtree_reproduced = treecompare.symmetric_difference(tree4, tree5)

    # 7. Normalize RF distance for unrooted trees
    num_taxa = len(tns)
    max_rf = 2 * (num_taxa - 3) if num_taxa >= 3 else 1
    normalized_rf_foldtree = rf_dist_foldtree / max_rf if max_rf > 0 else 0.0
    normalized_rf_aa_only = rf_dist_aa_only / max_rf if max_rf > 0 else 0.0
    normalized_rf_3di_our_pipeline = rf_dist_foldree_3di_our_pipeline / max_rf if max_rf > 0 else 0.0
    normalized_rf_esm3di_3di_our_pipeline = rf_dist_esm3di_3di_our_pipeline / max_rf if max_rf > 0 else 0.0
    normalized_rf_foldtree_reproduced = rf_dist_foldtree_reproduced / max_rf if max_rf > 0 else 0.0

    # 8. Output results
    print(f"--- RF Calculation for {prot} ---")
    print(f"Total Unique Shared Taxa Matched: {num_taxa}")
    print(f"Robinson-Foulds Distance (ESM3DI vs FoldTree (from structures)): {rf_dist_foldtree}")
    print(f"Robinson-Foulds Distance (ESM3DI vs MisfudAA Only): {rf_dist_aa_only}")
    print(f"Max Possible RF Distance: {max_rf}")
    print(f"Normalized Robinson-Foulds Distance (ESM3DI vs Misfud FoldTree): {normalized_rf_foldtree:.4f}")
    print(f"Normalized Robinson-Foulds Distance (ESM3DI vs Misfud AA Only): {normalized_rf_aa_only:.4f}")
    print(f"Normalized Robinson-Foulds Distance (Misfud FoldTree vs Misfud 3di with our pipeline): {normalized_rf_3di_our_pipeline:.4f}")
    print(f"Normalized Robinson-Foulds Distance (ESM3DI vs Misfud 3di with our pipeline): {normalized_rf_esm3di_3di_our_pipeline:.4f}")
    print(f"Normalized Robinson-Foulds Distance (Misfud from 3di vs from structures with our pipeline): {normalized_rf_foldtree_reproduced:.4f}")

--- RF Calculation for E ---
Total Unique Shared Taxa Matched: 248
Robinson-Foulds Distance (ESM3DI vs FoldTree (from structures)): 208
Robinson-Foulds Distance (ESM3DI vs MisfudAA Only): 240
Max Possible RF Distance: 490
Normalized Robinson-Foulds Distance (ESM3DI vs Misfud FoldTree): 0.4245
Normalized Robinson-Foulds Distance (ESM3DI vs Misfud AA Only): 0.4898
Normalized Robinson-Foulds Distance (Misfud FoldTree vs Misfud 3di with our pipeline): 0.3551
Normalized Robinson-Foulds Distance (ESM3DI vs Misfud 3di with our pipeline): 0.4245
Normalized Robinson-Foulds Distance (Misfud from 3di vs from structures with our pipeline): 0.0041
--- RF Calculation for E1 ---
Total Unique Shared Taxa Matched: 190
Robinson-Foulds Distance (ESM3DI vs FoldTree (from structures)): 132
Robinson-Foulds Distance (ESM3DI vs MisfudAA Only): 174
Max Possible RF Distance: 374
Normalized Robinson-Foulds Distance (ESM3DI vs Misfud FoldTree): 0.3529
Normalized Robinson-Foulds Distance (ESM3DI vs Misfud AA Only)

In [5]:
import dendropy
from dendropy.calculate import treecompare

def load_and_clean_newick(file_path):
    """Reads a tree file and removes '.pdb' from taxon identifiers."""
    with open(file_path, "r") as f:
        content = f.read()
    return content.replace(".pdb", "")

glycoproteins = ["E", "E1", "E2"]

for prot in glycoproteins:
    # 1. Establish a fresh shared namespace per glycoprotein
    tns = dendropy.TaxonNamespace()

    # 2. Read clean Newick strings
    misfud_foldtree = load_and_clean_newick(f"Foldtree/{prot.lower()}_foldtree_struct_tree.PP.nwk")
    misfud_aa_only = load_and_clean_newick(f"fn_3di_trees/refolded_fullglyco_{prot}_AA_famsa.fas.treefile")
    esm3di_unrooted = load_and_clean_newick(f"results/flaviviridae_fullglyco_{prot}/foldtree_struct_tree.PP.nwk")
    misfud_3di_our_pipeline = load_and_clean_newick(f"results/misfud_fastas/flaviviridae_fullglyco_{prot}/foldtree_struct_tree.PP.nwk")
    misfud_foldtree_reproduced = load_and_clean_newick(f"Foldtree_reproduced/{prot.lower()}_foldtree_struct_tree.PP.nwk")

    prostT5_unrooted = load_and_clean_newick(f"results/prostT5/{prot}_foldtree_struct_tree.PP.nwk") # new addition

    # 3. Load trees into DendroPy
    trees = []
    tree_sources = [misfud_foldtree, misfud_aa_only, esm3di_unrooted, misfud_3di_our_pipeline, misfud_foldtree_reproduced, prostT5_unrooted] # include prostT5_unrooted in the list
    
    for src in tree_sources:
        t = dendropy.Tree.get(
            data=src, 
            schema="newick", 
            taxon_namespace=tns,
            rooting="force-unrooted"
        )
        # Force unrooted state prior to tree restructuring
        t.is_rooted = False
        t.collapse_basal_bifurcation()
        t.encode_bipartitions()
        trees.append(t)

    tree1, tree2, tree3, tree4, tree5, tree6 = trees

    # 4. Taxa Sanity Check
    num_taxa = len(tns)
    for idx, t in enumerate(trees, 1):
        d=1
        #assert len(t.leaf_nodes()) == num_taxa, f"Taxon count mismatch in Tree {idx} for {prot}"

    # 5. Calculate unweighted Robinson-Foulds distances
    rf_dist_foldtree = treecompare.symmetric_difference(tree5, tree3)
    rf_dist_aa_only = treecompare.symmetric_difference(tree2, tree3)
    rf_dist_foldree_3di_our_pipeline = treecompare.symmetric_difference(tree1, tree4)
    rf_dist_esm3di_3di_our_pipeline = treecompare.symmetric_difference(tree3, tree4)
    rf_dist_foldtree_reproduced = treecompare.symmetric_difference(tree4, tree5)
    rf_dist_prostT5_foldtree = treecompare.symmetric_difference(tree6, tree5)
    rf_dist_prostT5_esm3di = treecompare.symmetric_difference(tree6, tree3)

    # 6. Normalize RF distance
    max_rf = 2 * (num_taxa - 3) if num_taxa >= 3 else 1
    
    norm_esm3di_vs_struct = rf_dist_foldtree / max_rf
    norm_esm3di_vs_aa = rf_dist_aa_only / max_rf
    norm_struct_vs_our_3di = rf_dist_foldree_3di_our_pipeline / max_rf
    norm_esm3di_vs_our_3di = rf_dist_esm3di_3di_our_pipeline / max_rf
    norm_struct_vs_reproduced = rf_dist_foldtree_reproduced / max_rf
    norm_prostT5_vs_struct = rf_dist_prostT5_foldtree / max_rf
    norm_prostT5_vs_esm3di = rf_dist_prostT5_esm3di / max_rf

    # 7. Output results
    print(f"\n=================== {prot} ===================")
    print(f"Total Unique Shared Taxa: {num_taxa} | Max Possible RF: {max_rf}")
    print(f"Norm. RF (ESM3DI vs Struct FoldTree):          {norm_esm3di_vs_struct:.4f}")
    print(f"Norm. RF (ESM3DI vs AA-Only Alignment):         {norm_esm3di_vs_aa:.4f}")
    print(f"Norm. RF (ESM3DI vs 3Di via Our Pipeline):      {norm_esm3di_vs_our_3di:.4f}")
    print(f"Norm. RF (Struct vs Reproduced 3Di Pipeline):  {norm_struct_vs_reproduced:.4f}")
    print(f"Norm. RF (Struct vs 3Di via Our Pipeline):      {norm_struct_vs_our_3di:.4f}")
    print(f"Norm. RF (ProstT5 vs Struct FoldTree):          {norm_prostT5_vs_struct:.4f}")
    print(f"Norm. RF (ProstT5 vs ESM3DI):                   {norm_prostT5_vs_esm3di:.4f}")


=================== E ===================
Total Unique Shared Taxa: 248 | Max Possible RF: 490
Norm. RF (ESM3DI vs Struct FoldTree):          0.4245
Norm. RF (ESM3DI vs AA-Only Alignment):         0.4898
Norm. RF (ESM3DI vs 3Di via Our Pipeline):      0.4245
Norm. RF (Struct vs Reproduced 3Di Pipeline):  0.0041
Norm. RF (Struct vs 3Di via Our Pipeline):      0.3551
Norm. RF (ProstT5 vs Struct FoldTree):          0.4367
Norm. RF (ProstT5 vs ESM3DI):                   0.4041

=================== E1 ===================
Total Unique Shared Taxa: 190 | Max Possible RF: 374
Norm. RF (ESM3DI vs Struct FoldTree):          0.3529
Norm. RF (ESM3DI vs AA-Only Alignment):         0.4652
Norm. RF (ESM3DI vs 3Di via Our Pipeline):      0.3529
Norm. RF (Struct vs Reproduced 3Di Pipeline):  0.0000
Norm. RF (Struct vs 3Di via Our Pipeline):      0.2193
Norm. RF (ProstT5 vs Struct FoldTree):          0.3583
Norm. RF (ProstT5 vs ESM3DI):                   0.4118

=================== E2 =================

## Conclusion

Not ideal to use this as for phylogenetic support in the manuscript

- paper results are not directly usable
- The distance which we would want to be small (ESM3di vs their 3di with our pipeline is quite large)


In [7]:
# compute RF distances between trees from the paper (Misfud et al. 2022)
import os
import pandas as pd


tree_dir = "fn_3di_trees/"

glycoprotein = 'E2'

tree_files = [f for f in os.listdir(tree_dir) if f.startswith(f"refolded_fullglyco_{glycoprotein}_") and f.endswith(".treefile")]

# generate distance matrix for all trees in the directory
distance_matrix = pd.DataFrame(index=tree_files, columns=tree_files)
for i, tree_file1 in enumerate(tree_files):
    tree1 = dendropy.Tree.get(data=load_and_clean_newick(os.path.join(tree_dir, tree_file1)), schema="newick", taxon_namespace=tns)
    tree1.encode_bipartitions()
    for j, tree_file2 in enumerate(tree_files):
        if i <= j:  # compute only upper triangle and diagonal
            tree2 = dendropy.Tree.get(data=load_and_clean_newick(os.path.join(tree_dir, tree_file2)), schema="newick", taxon_namespace=tns)
            tree2.encode_bipartitions()
            rf_distance = treecompare.symmetric_difference(tree1, tree2)
            distance_matrix.loc[tree_file1, tree_file2] = rf_distance
            distance_matrix.loc[tree_file2, tree_file1] = rf_distance  # symmetric



In [15]:
print(tree_files)

['refolded_fullglyco_E2_3di_AA_famsa_parts.nex.treefile', 'refolded_fullglyco_E2_AA_famsa.fas.treefile', 'refolded_fullglyco_E2_3di_t35_AA_3dit35_famsa_parts.nex.treefile', 'refolded_fullglyco_E2_3di_famsa_trim35.fas.treefile', 'refolded_fullglyco_E2_AA_famsa_3ditrim35.fas.treefile', 'refolded_fullglyco_E2_3di_famsa.fas.treefile']
